## Extracting panels of sentences
According to Bolubkasi et al., a total of "25 neurons, 33 random directions and 29 random sets of sentences" were annotated, each of these having exactly 10 sentences. We now extract and save panels for annotation, starting with the random sets of sentences.

In [23]:
import numpy as np
import pandas as pd

# load every dataset
ds_names = ['qqp', 'qnli', 'wiki', 'books']
datasets = {name: pd.read_parquet(f'../datasets/{name}.parquet') for name in ds_names}

# most complicated thing here is going to be
# just seeding the rng, rest is straightforward
rng = np.random.default_rng(1)

In [ ]:
# pairs (dataset name, how many panels to make)
# ideally, uniformly distributed, but 29 isn't divisible by 7
for name, K in [('qqp', 8), ('qnli', 7), ('wiki', 7), ('books', 7)]:
    # get dataset from name
    df = datasets[name]

    # extract K panels from it
    for k in range(K):
        # random sentence indexes
        idx = rng.choice(len(df), size=10, replace=False)
        # sliced dataframe with panel
        sents = df.iloc[idx]
        # save to disk
        sents.to_parquet(f'../panels/{name}_rand_{k}.parquet', index=False)


We now generate 33 random directions in the embedding space and get the top-activating sentences within them.

In [41]:
embeds = {name: np.load(f'../embeds/full/{name}_embed.npy') for name in ds_names}

# make a new one, the above rng was advanced
rng = np.random.default_rng(1)

In [42]:
# directions matrix, 33 entries, 768 components/vector
dirs = rng.standard_normal(tuple([33, 768]))
# normalize them
dirs /= np.linalg.norm(dirs, axis=1, keepdims=True)

In [ ]:
for name in ds_names:
    # sentences
    ds = datasets[name]
    # sentence embeddings
    ds_embed = np.array(embeds[name])
    # calculate scores for sentences on directions
    # this has shape (dataset size, 33 = 1 for each direction)
    score = ds_embed @ dirs.T 

    for k in range(33):
        # no comparator/reverse=True for argsort, sort by -score instead
        # best_idx = indices of best sentences for panel k
        best_idx = np.argsort(-score[:, k])[:10]
        # sliced dataframe with panel
        sents = ds.iloc[best_idx]
        # save to disk
        sents.to_parquet(f'../panels/{name}_dir_{k}.parquet', index=False)

Now for the last step, we make panels with the top-activating sentences for the neurons of BERT.

In [58]:
# make a new rng again
rng = np.random.default_rng(1)

# choose 25 neurons
neurons = rng.choice(768, size=25, replace=False)

When working with a random direction, we took a random vector in the 768-dimensional space of the embedding and multiplied the embedding of `[CLS]` with this random vector, albeit we did this for all of the sentences at once by working with matrices. This gives some weight to each component of the embedding. To obtain the top-activating sentences of a certain neuron `i`, only the component with index `i` must be considered, and all others must be given no weight. This is the same as multiplying with a vector that looks like `[1, 0, 0, 0, ...]` (for example, for `i=1`), which is to say, one of the vectors of the canonical basis of the embedding space.

What we have to do is, code-wise, very similar to the last cell. The authors perform matrix multiplication with a matrix of 25 random basis vectors, except we can avoid matrix multiplication altogether by simply looking at the component we are interested in in the embedded sentences.

In [ ]:
for name in ds_names:
    # sentences
    ds = datasets[name]
    # sentence embeddings
    ds_embed = np.array(embeds[name])

    for neur in neurons:
        # project to only look at the component with index neur
        score = ds_embed[:, neur]
        # same logic as 2 cells above
        best_idx = np.argsort(-score)[:10]
        # sliced dataframe with panel
        sents = ds.iloc[best_idx]
        # save to disk
        sents.to_parquet(f'../panels/{name}_neur_{k}.parquet', index=False)